# Kaggle T4 x2 pipeline (reference workflow, Tasks 1-6)

The submitted runs were trained on a Modal H100 (`modal_app.py`). This notebook runs the same code on Kaggle **GPU T4 x2**
with the guide's T4 settings: fp16 autocast + GradScaler, FP32 master weights, SFT micro-batch 2 x 16, DPO 1 x 32 with the
policy on `cuda:0` and the frozen reference on `cuda:1`.

In [ ]:
!nvidia-smi
!git clone --depth 1 https://github.com/<your-user>/small-model-alignment.git /kaggle/working/small-model-alignment
%cd /kaggle/working/small-model-alignment
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest "peft>=0.11" "trl>=0.13"
!python -m pip install -q -e . --no-deps
import torch, transformers, datasets
print("torch", torch.__version__, "transformers", transformers.__version__, "datasets", datasets.__version__)
print([torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
!PYTHONPATH=. python -m pytest -q tests
!python -m sma.data --task all --out_dir outputs/data

In [ ]:
!python -m sma.train_sft --config configs/sft.yaml --precision fp16 --per_device_batch_size 2 --gradient_accumulation_steps 16 --gradient_checkpointing true

In [ ]:
!python -m sma.train_dpo --config configs/dpo_hh.yaml --precision fp16 --per_device_batch_size 1 --gradient_accumulation_steps 32 --eval_batch_size 4 --reference_device cuda:1
!python -m sma.train_dpo --config configs/dpo_uf.yaml --precision fp16 --per_device_batch_size 1 --gradient_accumulation_steps 32 --eval_batch_size 4 --reference_device cuda:1
!python -m sma.train_lora_dpo --config configs/lora_dpo.yaml

In [ ]:
!python -m sma.generate --model_path outputs/sft/best --generator sft --eval_path eval_sets/alpaca_eval_300_seed42.json --output_path outputs/generations/sanity/sft.json --limit 5 --dtype fp16
# then see modal_app.py::generate for the full list of generation + build_pairs commands
!cd /kaggle/working && zip -qr sma_outputs.zip small-model-alignment/outputs -x "*/best/*" "*/final/*"